In [17]:
import pandas as pd

def build_team_winrate_std_csv(in_path: str, out_path: str) -> pd.DataFrame:
    """
    Rank teams within league by std desc, tie break done through alphabetical order to avoid same ranks

    Output columns: league, team, win_rate, std_win_rate, rank
      - win_rate = mean of season win rates across seasons
      - std_win_rate = std of season win rates across seasons (population)
    """
    df = pd.read_csv(in_path)
    need = {"season","date","league","team1","team2","result","score1","score2"}
    miss = need - set(df.columns)
    if miss:
        raise ValueError(f"Missing columns: {sorted(miss)}")

    # Computes each teams's season win rate
    home = pd.DataFrame({"league": df["league"], "season": df["season"],
                         "team": df["team1"], "win": (df["result"] == 1).astype(int)})
    away = pd.DataFrame({"league": df["league"], "season": df["season"],
                         "team": df["team2"], "win": (df["result"] == -1).astype(int)})
    long = pd.concat([home, away], ignore_index=True)

    season_rates = (long
        .groupby(["league","season","team"], as_index=False)
        .agg(season_win_rate=("win","mean"))
    )

    # Group by league and team to get each team's average season win_rate and its across seasons std
    team_across = (season_rates
        .groupby(["league","team"], as_index=False)
        .agg(
            win_rate=("season_win_rate","mean"),
            std_win_rate=("season_win_rate", lambda x: x.std(ddof=0)),
        )
    )

    # Rank each team within league: std desc, then alphabeticaly to avoid same ranks
    def _rank_block(g):
        g = g.sort_values(["std_win_rate","team"], ascending=[False, True]).reset_index(drop=True)
        g["rank"] = g.index + 1
        return g

    ranked = (team_across
        .groupby("league", group_keys=True)
        .apply(_rank_block)
        .reset_index(drop=True)
        .sort_values(["league","rank","team"])
        .reset_index(drop=True)
    )

    out = ranked[["league","team","win_rate","std_win_rate","rank"]]
    out.to_csv(out_path, index=False)
    print(f"Wrote: {out_path}  (rows: {len(out):,})")
    return out

In [18]:
# run to generate csv files

# Actual
actual = build_team_winrate_std_csv(
    in_path="../../data/us_leagues/csv/game_by_game/actual/us_combined_data.csv",
    out_path="../../data/us_leagues/csv/standard_deviation/actual/sd_win_rate_us_actual.csv",)

# Pure Skill
skill = build_team_winrate_std_csv(
    in_path="../../data/us_leagues/csv/game_by_game/pure_skill/us_combined_pure_skill.csv",
    out_path="../../data/us_leagues/csv/standard_deviation/pure_skill/sd_win_rate_us_pure_skill.csv",)

# Pure Luck (home bias v1)
luck = build_team_winrate_std_csv(
    in_path="../../data/us_leagues/csv/game_by_game/pure_luck/us_coin_flip_home_bias_v1.csv",
    out_path="../../data/us_leagues/csv/standard_deviation/pure_luck/sd_win_rate_us_pure_luck_home_bias.csv",)

Wrote: ../../data/us_leagues/csv/standard_deviation/actual/sd_win_rate_us_actual.csv  (rows: 128)
Wrote: ../../data/us_leagues/csv/standard_deviation/pure_skill/sd_win_rate_us_pure_skill.csv  (rows: 128)
Wrote: ../../data/us_leagues/csv/standard_deviation/pure_luck/sd_win_rate_us_pure_luck_home_bias.csv  (rows: 128)
